In [68]:
import os
import pandas as pd
import numpy as np

def bern(p):
    return np.random.choice([True, False], p=[p, 1-p])

def process_file(input_path, output_path, p=0.3):
    """Обрабатывает один файл и сохраняет результат"""
    try:
        data = pd.read_csv(input_path)
        
        # Добавляем пропуски во все колонки кроме 'Target'
        for col in data.columns:
            if col != 'Target':
                mask = data[col].apply(lambda x: bern(p))
                data.loc[mask, col] = np.nan
        
        # Сохраняем в новую папку
        data.to_csv(output_path, index=False)
        print(f"Обработан: {os.path.basename(input_path)}")
        return True
    except Exception as e:
        print(f"Ошибка в файле {input_path}: {str(e)}")
        return False

def process_folder(input_folder, output_folder, p=0.3):
    """Обрабатывает все CSV-файлы в папке"""
    # Создаем целевую папку если её нет
    os.makedirs(output_folder, exist_ok=True)
    
    processed_files = 0
    for filename in os.listdir(input_folder):
        if filename.endswith('.csv'):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            if process_file(input_path, output_path, p):
                processed_files += 1
    
    print(f"\nГотово! Обработано файлов: {processed_files}")
    print(f"Результаты сохранены в: {os.path.abspath(output_folder)}")

# Укажите пути
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MCAR_30_openML_datasets')

# Запуск обработки с вероятностью пропусков 30%
process_folder(input_folder, output_folder)

Обработан: 2dplanes_binclass_binaryClass.csv
Обработан: 2dplanes_regression_y.csv
Обработан: abalone_binclass_binaryClass.csv
Обработан: ailerons_binclass_binaryClass.csv
Обработан: Ailerons_regression_goal.csv
Обработан: analcatdata_supreme_binclass_binaryClass.csv
Обработан: analcatdata_supreme_regression_Log_exposure.csv
Обработан: bank32nh_regression_rej.csv
Обработан: bank8FM_binclass_binaryClass.csv
Обработан: bank8FM_regression_rej.csv
Обработан: BNG(breast-w)_binclass_Class.csv
Обработан: coil2000_regression_CARAVAN.csv
Обработан: cpu_act_binclass_binaryClass.csv
Обработан: cpu_act_regression_usr.csv
Обработан: cpu_small_binclass_binaryClass.csv
Обработан: cpu_small_regression_usr.csv
Обработан: delta_ailerons_binclass_binaryClass.csv
Обработан: delta_elevators_binclass_binaryClass.csv
Обработан: delta_elevators_regression_Se.csv
Обработан: electricity_binclass_class.csv
Обработан: elevators_regression_Goal.csv
Обработан: fried_regression_Y.csv
Обработан: fri_c0_1000_10_regress

In [4]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def bern(p):
    """Генерирует True с вероятностью p (без изменений)"""
    return np.random.choice([True, False], p=[p, 1-p])

def calculate_p_sigmoid(col_values, other_values, steepness=1.0):
    """
    Сигмоидальная вероятность пропуска на основе других признаков
    steepness=0.3 делает зависимость более плавной (меньше пропусков)
    """
    # Вычисляем среднее значение других признаков
    mean_other = np.nanmean(other_values, axis=1)
    mean_other = np.nan_to_num(mean_other, nan=0.0)  # Заменяем NaN на 0
    
    # Нормализуем значения текущего столбца
    col_normalized = (col_values - np.nanmean(col_values)) / (np.nanstd(col_values) + 1e-8)
    
    # Сигмоида от разницы между текущим значением и средним других признаков
    z = steepness * (col_normalized - mean_other)
    z = np.clip(z, -500, 500)  # Защита от переполнения
    return 1 / (1 + np.exp(-z))

def MAR(input_path, output_path):
    try:
        data = pd.read_csv(input_path)
        
        if 'Target' not in data.columns:
            raise ValueError("Отсутствует колонка 'Target'")
        
        features = [col for col in data.columns if col != 'Target']
        data_np = data[features].values.astype(float)
        
        for j, col in enumerate(features):
            other_cols = [i for i in range(len(features)) if i != j]
            other_values = data_np[:, other_cols]
            
            p_values = calculate_p_sigmoid(data_np[:, j], other_values)
            
            # Используем bern(p) для каждого элемента
            mask = np.array([bern(p) for p in p_values])
            data.loc[mask, col] = np.nan
        
        data.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False

def process_folder_MAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            if MAR(input_path, output_path):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установка seed для воспроизводимости
np.random.seed(42)

# Пути к данным
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MAR_sigmoid_processed')

# Запуск обработки
process_folder_MAR(input_folder, output_folder)


True

In [2]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def bern(p):
    """Генерирует True с вероятностью p (без изменений)"""
    return np.random.choice([True, False], p=[p, 1-p])

'''def calculate_p_MNAR(col_values, eps=1e-8):
    """Векторизованный расчет вероятностей для всего столбца"""
    max_abs = np.max(np.abs(col_values)) + eps
    return np.clip(np.abs(col_values) / max_abs, 0.0, 1.0)
'''
'''  
def calculate_p_MNAR(col_values, steepness=1.0, midpoint=0.0):
    values = col_values.astype(float)
    # Нормализация
    mean = np.nanmean(values)
    std = np.nanstd(values)
    if std > 1e-8:  # Практически ноль
         values = (values - mean) / std
    """Векторизованный расчет вероятностей для всего столбца"""
    # Логистическая функция для MNAR
    z = steepness * (values - midpoint)
    z = np.clip(z, -500, 500)  # Численная стабильность
    return 1 / (1 + np.exp(-z))
'''
def calculate_p_MNAR(col_values, steepness=0.3, midpoint=0.0):
    """Векторизованный расчет вероятностей для всего столбца"""
    z = steepness * (col_values - midpoint)
    z = np.clip(z, -500, 500) 
    return 1 / (1 + np.exp(-z))

def MNAR(input_path, output_path):
    try:
        data = pd.read_csv(input_path)
        
        for col in [c for c in data.columns if c != 'Target']:
            # Векторизованный расчет вероятностей
            p_values = calculate_p_MNAR(data[col].values)
            
            # Применяем bern(p) ко всем значениям через векторную функцию
            mask = np.array([bern(p) for p in p_values])
            data[col] = np.where(mask, np.nan, data[col])
        
        data.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False

def process_folder_MNAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            if MNAR(input_path, output_path):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установите seed для воспроизводимости
#np.random.seed(42)

# Запуск обработки
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MNAR_openML_datasets_optimized')
process_folder_MNAR(input_folder, output_folder)

Обработка файлов: 100%|████████████████████████████████████████████████████████████████| 96/96 [21:36<00:00, 13.51s/it]


Готово! Обработано 96/96 файлов
Результаты в: C:\Users\nikit\summer\openML_datasets\MNAR_openML_datasets_optimized


In [7]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def bern(p):
    return np.random.choice([True, False], p=[p, 1-p])

def maska(col, procent):
    """
    Постепенно заменяет элементы на np.nan, пока не достигнет нужного процента пропусков.
    Если столбец закончился — начинает с начала.
    """
    n = len(col)
    target_n = int(round(n * procent)) # сколько пропусков надо сделать
    data = col.copy()
    nan_count = 0
    index = 0
    
    while nan_count < target_n:
        if not pd.isna(data.loc[index]):
            if bern(procent):  # Вероятность p для замены
                data.loc[index] = np.nan
                nan_count += 1
        index = (index + 1) % n  # Циклический переход к следующему элементу
    
    return data.isna()  # Возвращаем булеву маску

def MCAR(input_path, output_path, procent):
    """Обрабатывает один файл и сохраняет результат"""
    try:
        data = pd.read_csv(input_path)
        
        # Добавляем пропуски во все колонки кроме 'Target'
        for col in data.columns:
            if col != 'Target':
                mask = maska(data[col], procent)
                data.loc[mask, col] = np.nan
                # 2. Перемешиваем ВСЕ строки таблицы
                data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        # Сохраняем в новую папку
        data.to_csv(output_path, index=False)
        #print(f"Обработан: {os.path.basename(input_path)}")
        return True
    except Exception as e:
        print(f"Ошибка в файле {input_path}: {str(e)}")
        return False

def process_folder(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            procent = 0.5
            if MCAR(input_path, output_path, procent):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установка seed для воспроизводимости
np.random.seed(42)

# Пути к данным
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MCAR_50')

# Запуск обработки
process_folder(input_folder, output_folder)

Обработка файлов: 100%|██████████████████████████████████████████████████████████████| 96/96 [1:09:22<00:00, 43.36s/it]


Готово! Обработано 96/96 файлов
Результаты в: C:\Users\nikit\summer\openML_datasets\MCAR_50


In [17]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def bern(p):
    return np.random.choice([True, False], p=[p, 1-p])

def maska_MNAR(col, procent):
    n = len(col)
    target_n = int(round(n * procent))
    if target_n == 0:
        return pd.Series(False, index=col.index)
    
    # Учитываем только не-NaN значения
    valid_mask = ~col.isna()
    valid_values = col[valid_mask].abs()
    
    # Полная защита от всех крайних случаев:
    if len(valid_values) == 0 or valid_values.sum() == 0:
        if len(valid_values) == 0:  # Если вообще нет валидных значений
            return pd.Series(False, index=col.index)
        # Для нулевых значений используем равномерное распределение
        replace_indices = np.random.choice(
            valid_values.index,
            size=min(target_n, len(valid_values)),
            replace=False
        )
    else:
        # Нормальный случай с ненулевыми значениями
        p = valid_values / valid_values.sum()
        try:
            replace_indices = np.random.default_rng(seed=42).choice(
                valid_values.index,
                size=min(target_n, len(valid_values)),
                replace=False,
                p=p
            )
        except ValueError:
            # Если возникли проблемы с вероятностями
            replace_indices = np.random.choice(
                valid_values.index,
                size=min(target_n, len(valid_values)),
                replace=False
            )
    
    # Создаем маску
    mask = pd.Series(False, index=col.index)
    if len(valid_values) > 0:  # Добавляем эту проверку
        mask.loc[replace_indices] = True
    return mask

def MNAR(input_path, output_path, procent):
    try:
        data = pd.read_csv(input_path)
        
        for col in [c for c in data.columns if c != 'Target']:
            mask = maska_MNAR(data[col], procent)
            data.loc[mask, col] = np.nan
            # 2. Перемешиваем ВСЕ строки таблицы
            data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        data.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False

def process_folder_MNAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            if MNAR(input_path, output_path, procent = 0.5):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установите seed для воспроизводимости
np.random.seed(42)

# Запуск обработки
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MNAR_50')
process_folder_MNAR(input_folder, output_folder)

Обработка файлов: 100%|████████████████████████████████████████████████████████████████| 96/96 [07:09<00:00,  4.47s/it]


Готово! Обработано 96/96 файлов
Результаты в: C:\Users\nikit\summer\openML_datasets\MNAR_50


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def bern(p):
    return np.random.choice([True, False], p=[p, 1-p])
'''
def maska_MAR(other_values, procent):
    n = len(other_values)
    target_n = int(round(n * procent))
    
    valid_values = other_values.abs()
    col = valid_values.sum(axis = 1)
    # получаем столбец вероятностей
    if col.sum() == 0:
        p = np.ones_like(col) / len(col)  # Вероятности 1/n для всех элементов
    else:
        p = col / col.sum()  # Нормализованные веса
    # p = col / col.sum()
    # Выбираем индексы для замены
    rng = np.random.default_rng(seed=42)
    replace_indices = rng.choice(
        col.index,
        size=min(target_n, len(col)),
        replace=False,
        p=p
    )
        
    # Создаем маску
    mask = pd.Series(False, index=col.index)
    if len(valid_values) > 0:  # Добавляем эту проверку
        mask.loc[replace_indices] = True
    return mask
'''

def maska_MAR(other_values, procent):
    n = len(other_values)
    target_n = int(round(n * procent))
    
    valid_values = other_values.abs()
    col = valid_values.sum(axis=1)
    
    # Если все значения нулевые — делаем равномерное распределение
    p = col / col.sum()  # Нормализованные веса
    
    # Сколько элементов можно выбрать (не больше, чем ненулевых вероятностей)
    #available_non_zero = np.count_nonzero(p)
    #actual_size = min(target_n, available_non_zero)
    actual_size = target_n
    # Если нет доступных элементов — возвращаем маску из False
    #if actual_size == 0:
    #    return pd.Series(False, index=col.index)
    
    # Выбираем индексы для замены
    rng = np.random.default_rng(seed=42)
    replace_indices = rng.choice(
        col.index,
        size=actual_size,
        replace=False,
        p=p
    )
    
    # Создаём маску
    mask = pd.Series(False, index=col.index)
    mask.loc[replace_indices] = True
    
    return mask

def MAR(input_path, output_path, procent):
    try:
        data = pd.read_csv(input_path)
        dataset = data.copy()
        if 'Target' not in data.columns:
            raise ValueError("Отсутствует колонка 'Target'")
        
        features = [col for col in data.columns if col != 'Target']
        
        for j, col in enumerate(features):
            # Выбираем все колонки кроме текущей
            other_cols = [f for f in features if f != col]
            other_values = data[other_cols]  # Оставляем как DataFrame
            
            mask = maska_MAR(other_values, procent)
            dataset.loc[mask, col] = np.nan
            # 2. Перемешиваем ВСЕ строки таблицы
            data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        dataset.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False

def process_folder_MAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            procent = 0.05
            if MAR(input_path, output_path, procent):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установите seed для воспроизводимости
np.random.seed(42)

# Запуск обработки
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'MAR_5')
process_folder_MAR(input_folder, output_folder)

Обработка файлов:  98%|██████████████████████████████████████████████████████████████▋ | 94/96 [20:25<00:04,  2.22s/it]

In [4]:
data = pd.read_csv('2dplanes_binclass_binaryClass.csv')
print(data.head())
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
print(data.head())

    x1   x2   x3   x4   x5   x6   x7   x8   x9  x10  Target
0  1.0  1.0  1.0  1.0 -1.0  0.0  1.0  0.0 -1.0  1.0       0
1 -1.0  0.0 -1.0 -1.0  1.0  1.0  1.0  0.0 -1.0  0.0       0
2  1.0  0.0  1.0  1.0 -1.0 -1.0  0.0  1.0  1.0  1.0       0
3  1.0  0.0  0.0  1.0  0.0 -1.0  1.0  1.0  1.0 -1.0       0
4 -1.0  0.0 -1.0 -1.0 -1.0 -1.0 -1.0  1.0  1.0 -1.0       1
    x1   x2   x3   x4   x5   x6   x7   x8   x9  x10  Target
0 -1.0  1.0  1.0  0.0 -1.0  0.0  1.0  1.0  0.0  0.0       1
1  1.0 -1.0  1.0 -1.0  1.0  1.0  1.0 -1.0  0.0  1.0       0
2 -1.0  0.0  0.0 -1.0  1.0 -1.0  1.0  0.0  1.0  1.0       1
3 -1.0  1.0 -1.0  1.0 -1.0 -1.0  1.0  0.0 -1.0 -1.0       1
4 -1.0  0.0  1.0 -1.0  1.0  1.0  1.0  0.0 -1.0 -1.0       0


In [1]:
import pandas as pd
import numpy as np

def maska_MNAR(col, procent):
    n = len(col)
    target_n = int(round(n * procent))
    if target_n == 0:
        return pd.Series(False, index=col.index)
    
    # Учитываем только не-NaN значения
    valid_mask = ~col.isna()
    valid_values = col[valid_mask].abs()
    
    # Полная защита от всех крайних случаев:
    if len(valid_values) == 0 or valid_values.sum() == 0:
        if len(valid_values) == 0:  # Если вообще нет валидных значений
            return pd.Series(False, index=col.index)
        # Для нулевых значений используем равномерное распределение
        replace_indices = np.random.choice(
            valid_values.index,
            size=min(target_n, len(valid_values)),
            replace=False
        )
    else:
        # Нормальный случай с ненулевыми значениями
        p = valid_values / valid_values.sum()
        try:
            replace_indices = np.random.default_rng(seed=42).choice(
                valid_values.index,
                size=min(target_n, len(valid_values)),
                replace=False,
                p=p
            )
        except ValueError:
            # Если возникли проблемы с вероятностями
            replace_indices = np.random.choice(
                valid_values.index,
                size=min(target_n, len(valid_values)),
                replace=False
            )
    
    # Создаем маску
    mask = pd.Series(False, index=col.index)
    if len(valid_values) > 0:  # Добавляем эту проверку
        mask.loc[replace_indices] = True
    return mask

data = pd.read_csv('2dplanes_binclass_binaryClass.csv')
print(maska_MNAR(data['x1'], 0.3))

0         True
1        False
2        False
3        False
4         True
         ...  
40763    False
40764    False
40765     True
40766    False
40767     True
Length: 40768, dtype: bool


In [11]:
data = pd.read_csv('2dplanes_binclass_binaryClass.csv')
features = [col for col in data.columns if col != 'Target']
data_np = data[features].values.astype(float)
print(type(data))

<class 'pandas.core.frame.DataFrame'>


In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def MNAR(input_path, output_path, procent):
    try:
        data = pd.read_csv(input_path)
        
        for col in [c for c in data.columns if c != 'Target']:
            data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        data.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False

def process_folder_MNAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            
            if MNAR(input_path, output_path, procent = 0.5):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установите seed для воспроизводимости
np.random.seed(42)

# Запуск обработки
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'change')
process_folder_MNAR(input_folder, output_folder)

Обработка файлов: 100%|████████████████████████████████████████████████████████████████| 96/96 [10:42<00:00,  6.69s/it]


Готово! Обработано 96/96 файлов
Результаты в: C:\Users\nikit\summer\openML_datasets\change


In [2]:
def MCAR(input_path, output_path, procent):
    """Обрабатывает один файл и сохраняет результат"""
    try:
        data = pd.read_csv(input_path)
        
        # Добавляем пропуски во все колонки кроме 'Target'
        for col in data.columns:
            if col != 'Target':
                
                data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        # Сохраняем в новую папку
        data.to_csv(output_path, index=False)
        #print(f"Обработан: {os.path.basename(input_path)}")
        return True
    except Exception as e:
        print(f"Ошибка в файле {input_path}: {str(e)}")
        return False

def process_folder(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            procent = 0.5
            if MCAR(input_path, output_path, procent):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установка seed для воспроизводимости
np.random.seed(42)

# Пути к данным
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'change1')

# Запуск обработки
process_folder(input_folder, output_folder)

Обработка файлов:  57%|████████████████████████████████████▋                           | 55/96 [00:32<00:24,  1.71it/s]


KeyboardInterrupt: 

In [3]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

def MAR(input_path, output_path, procent):
    try:
        data = pd.read_csv(input_path)
        dataset = data.copy()
        if 'Target' not in data.columns:
            raise ValueError("Отсутствует колонка 'Target'")
        
        features = [col for col in data.columns if col != 'Target']
        
        for j, col in enumerate(features):
            data = data.sample(frac=1, random_state=42).reset_index(drop=True)
        
        dataset.to_csv(output_path, index=False)
        return True
    except Exception as e:
        print(f"Ошибка в {os.path.basename(input_path)}: {str(e)}")
        return False


def process_folder_MAR(input_folder, output_folder):
    """Обрабатывает все CSV-файлы с прогресс-баром"""
    os.makedirs(output_folder, exist_ok=True)
    files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]
    
    with tqdm(total=len(files), desc="Обработка файлов") as pbar:
        processed_files = 0
        for filename in files:
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)
            procent = 0.05
            if MAR(input_path, output_path, procent):
                processed_files += 1
            pbar.update(1)
    
    print(f"\nГотово! Обработано {processed_files}/{len(files)} файлов")
    print(f"Результаты в: {os.path.abspath(output_folder)}")

# Установите seed для воспроизводимости
np.random.seed(42)

# Запуск обработки
input_folder = r'C:\Users\nikit\summer\openML_datasets'
output_folder = os.path.join(input_folder, 'change2')
process_folder_MAR(input_folder, output_folder)


Обработка файлов: 100%|████████████████████████████████████████████████████████████████| 96/96 [10:44<00:00,  6.71s/it]


Готово! Обработано 96/96 файлов
Результаты в: C:\Users\nikit\summer\openML_datasets\change2
